# 03 — Split folds

Creates leakage-safe chronological development folds and a physically isolated test set for each station.

**Inputs:** `data/interim/*.parquet`
**Outputs:** `data/processed/{station_id}_train_splits.parquet`, `{station_id}_test.parquet`, and `{station_id}_split_metadata.json`

In [ ]:
## Setup

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_CV_FOLDS,
    STATION_IDS,
    TEST_FRACTION,
)
from src.fetch_data import summarize_failures
from src.split_folds import SplitConfig, split_station_frame, write_split_artifacts

INTERIM_DIR = Path("data/interim")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
failures = {}
split_config = SplitConfig(
    test_fraction=TEST_FRACTION,
    n_folds=N_CV_FOLDS,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    horizon_hours=FORECAST_HORIZON_HOURS,
)

## Chronological split contract

After sorting, each station must already be a unique, contiguous hourly UTC series. The first 80% of rows form development data and the final 20% remain physically isolated in a separate test artifact. Development starts with a 50% training window and uses five expanding folds. Each validation block has a 24-hour embargo before it, and validation anchors stop 24 hours before the block ends so targets never enter the next block.

`target_valid` requires all water-level targets from `t+1` through `t+24` to be present and non-imputed. It is computed independently inside the development and test artifacts, so no target can cross their physical boundary.

In [ ]:
station_summaries = []
fold_summaries = []

for station_id in tqdm(STATION_IDS, desc="Splitting", unit="station"):
    source_path = INTERIM_DIR / f"{station_id}_hourly.parquet"

    try:
        station = pd.read_parquet(source_path)
        split = split_station_frame(
            station,
            station_id=station_id,
            config=split_config,
        )
        metadata = write_split_artifacts(
            split,
            station_id=station_id,
            source_path=source_path,
            output_dir=PROCESSED_DIR,
        )
        station_summaries.append(
            {
                "station_id": station_id,
                "source_rows": len(station),
                "development_rows": len(split.train),
                "test_rows": len(split.test),
                "development_targets_valid": int(split.train["target_valid"].sum()),
                "test_targets_valid": int(split.test["target_valid"].sum()),
            }
        )
        for fold in metadata["folds"]:
            fold_summaries.append(
                {
                    "station_id": station_id,
                    "fold": fold["fold"],
                    **fold["role_counts"],
                    "eligible_validation": fold["eligible_counts"]["validation"],
                }
            )
        print(f"Saved split artifacts and manifest for {station_id}")
    except Exception as error:  # noqa: BLE001 - aggregate station failures
        failures[str(source_path)] = error
        print(f"Failed {source_path}: {error}")

In [ ]:
if station_summaries:
    display(pd.DataFrame(station_summaries))
    display(pd.DataFrame(fold_summaries))
else:
    print("No split summaries are available to display.")

In [ ]:
if failures:
    raise RuntimeError(summarize_failures(failures))

print("Chronological fold splitting completed successfully.")